# 1/2 · Création de la base de connaissances (RAG) : radiologie

Ce notebook lit `radiologie.txt`, découpe le texte, calcule les embeddings avec `bge-m3` et enregistre la base `chroma_db` dans **Google Drive**.

**À exécuter une seule fois**, de haut en bas. Ensuite, utilisez le notebook `chatbot`.

## 1. Installer et démarrer Ollama

In [ ]:
# Installe zstd (nécessaire pour décompresser l'archive d'Ollama)
!sudo apt-get install -y zstd
# Télécharge et installe Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Lance le serveur Ollama en arrière-plan (systemd n'existe pas dans Colab)
!nohup ollama serve > ollama.log 2>&1 &

# Attend que le serveur soit prêt
import time
time.sleep(5)

In [ ]:
# Modèle d'embeddings
!ollama pull bge-m3
!ollama list

## 2. Installer les bibliothèques Python

In [ ]:
!pip install -q langchain-core langchain-text-splitters langchain-ollama langchain-chroma

## 2. Connecter Google Drive

Les deux notebooks tournent sur des machines Colab **séparées**. Google Drive sert de dossier commun :

```
Mon Drive/
└── radiologie_rag/
    ├── data/
    │   └── radiologie.txt
    └── chroma_db/          ← créé par creation_base, lu par chatbot
```

In [ ]:
from google.colab import drive
from pathlib import Path

# Une fenêtre demande l'autorisation d'accéder à votre Drive : acceptez
drive.mount("/content/drive")

# Dossier commun aux deux notebooks (dans Mon Drive)
DOSSIER_DRIVE = Path("/content/drive/MyDrive/radiologie_rag")
DATA_DRIVE = DOSSIER_DRIVE / "data"
CHROMA_DRIVE = DOSSIER_DRIVE / "chroma_db"

# Crée les dossiers s'ils n'existent pas
DATA_DRIVE.mkdir(parents=True, exist_ok=True)
print("Dossier Drive prêt :", DOSSIER_DRIVE)

## 3. Ajouter le document

Si `radiologie.txt` est déjà dans `radiologie_rag/data/` sur votre Drive, il est utilisé directement. Sinon, une fenêtre vous demande de l'envoyer.

In [ ]:
fichier = DATA_DRIVE / "radiologie.txt"

if fichier.exists():
    print("radiologie.txt trouvé dans Drive.")
else:
    from google.colab import files
    print("radiologie.txt introuvable : choisissez le fichier.")
    televerse = files.upload()
    for nom in televerse:
        Path(nom).replace(DATA_DRIVE / nom)
    print("Fichier enregistré dans :", DATA_DRIVE)

print("Contenu de data/ :", [p.name for p in DATA_DRIVE.iterdir()])

## 4. Charger le document

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

chemin = DATA_DRIVE / "radiologie.txt"

texte = chemin.read_text(encoding="utf-8")

document = Document(
    page_content=texte,
    metadata={
        "source": "radiologie.txt",
        "service": "Radiologie",
        "statut": "exemple_fictif"
    }
)

print("Document chargé avec succès.")

## 5. Découper le document en petits passages

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = splitter.split_documents([document])

print("Nombre de passages :", len(chunks))

## 6. Charger le modèle d'embeddings

In [ ]:
embeddings = OllamaEmbeddings(
    model="bge-m3"
)

print("Modèle d'embeddings chargé.")

## 7. Créer la base vectorielle ChromaDB

La base est d'abord construite sur le disque local de Colab (`/content/chroma_db`), car ChromaDB fonctionne mal directement sur Drive. Elle est ensuite copiée dans Drive.

Le dossier local est supprimé avant la création, ce qui évite les doublons si vous relancez la cellule.

In [ ]:
import shutil

CHROMA_LOCAL = "/content/chroma_db"

# Repart de zéro (évite les passages en double)
shutil.rmtree(CHROMA_LOCAL, ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_LOCAL,
    collection_name="radiologie"
)

print("Base de connaissances créée avec succès !")

## 8. Sauvegarder la base dans Google Drive

In [ ]:
# Remplace l'ancienne version dans Drive, puis copie la nouvelle
shutil.rmtree(CHROMA_DRIVE, ignore_errors=True)
shutil.copytree(CHROMA_LOCAL, CHROMA_DRIVE)

print("Base copiée dans Drive :", CHROMA_DRIVE)
print("Contenu :", [p.name for p in CHROMA_DRIVE.iterdir()])

## 9. Test rapide : rechercher un passage

In [ ]:
resultats = vectorstore.similarity_search("horaires", k=2)

for i, r in enumerate(resultats, 1):
    print(f"--- Passage {i} ---")
    print(r.page_content)
    print()